# Task 017: pnp_cassi — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Snapshot compressive spectral imaging (CASSI) using plug-and-play ADMM with denoiser

**Paper**: Deep plug-and-play priors for spectral snapshot compressive imaging
**Repository**: https://github.com/zsm1211/PnP-CASSI

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 21.76 dB
- **SSIM**: N/A


### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import os
import math
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`, `backward_operator`, `shift`, `shift_back`, `TV_denoiser`

In [ ]:
def load_and_preprocess_data(matfile, maskfile, step):
    """
    Load and preprocess data for CASSI reconstruction.
    Returns truth, mask_3d, and simulated measurement.
    """
    # Load Truth
    truth = sio.loadmat(matfile)['img']
    # Normalize truth to [0, 1] if not already
    if truth.max() > 1:
        truth = truth / 255.0
        
    r, c, nC = truth.shape
    
    # Load and Prepare Mask
    mask256 = sio.loadmat(maskfile)['mask']
    mask = np.zeros((r, c + step * (nC - 1)))
    mask_3d = np.tile(mask[:, :, np.newaxis], (1, 1, nC))
    
    for i in range(nC):
        mask_3d[:, i:i+256, i] = mask256
    
    # Shift truth for measurement simulation
    row, col, nC_truth = truth.shape
    truth_shift = np.zeros((row, col + (nC_truth - 1) * step, nC_truth))
    for i in range(nC_truth):
        truth_shift[:, i * step:i * step + col, i] = truth[:, :, i]
    
    # Generate Measurement (Simulation)
    meas = np.sum(mask_3d * truth_shift, axis=2)
    
    return truth, mask_3d, meas

def backward_operator(y, Phi):
    """
    Transpose of the forward model.
    """
    return np.multiply(np.repeat(y[:, :, np.newaxis], Phi.shape[2], axis=2), Phi)

def shift(inputs, step):
    """
    Simulate the dispersion effect (spatial shift).
    """
    row, col, nC = inputs.shape
    output = np.zeros((row, col + (nC - 1) * step, nC))
    for i in range(nC):
        output[:, i * step:i * step + col, i] = inputs[:, :, i]
    return output

def shift_back(inputs, step):
    """
    Reverse the dispersion effect.
    """
    row, col, nC = inputs.shape
    inputs_copy = inputs.copy()
    for i in range(nC):
        inputs_copy[:, :, i] = np.roll(inputs_copy[:, :, i], (-1) * step * i, axis=1)
    output = inputs_copy[:, 0:col - step * (nC - 1), :]
    return output

def TV_denoiser(x, _lambda, n_iter_max):
    """
    Total Variation Denoiser (Chambolle's algorithm).
    """
    dt = 0.25
    N = x.shape
    idx = np.arange(1, N[0] + 1)
    idx[-1] = N[0] - 1
    iux = np.arange(-1, N[0] - 1)
    iux[0] = 0
    ir = np.arange(1, N[1] + 1)
    ir[-1] = N[1] - 1
    il = np.arange(-1, N[1] - 1)
    il[0] = 0
    p1 = np.zeros_like(x)
    p2 = np.zeros_like(x)
    divp = np.zeros_like(x)

    for i in range(n_iter_max):
        z = divp - x * _lambda
        z1 = z[:, ir, :] - z
        z2 = z[idx, :, :] - z
        denom_2d = 1 + dt * np.sqrt(np.sum(z1 ** 2 + z2 ** 2, 2))
        denom_3d = np.tile(denom_2d[:, :, np.newaxis], (1, 1, N[2]))
        p1 = (p1 + dt * z1) / denom_3d
        p2 = (p2 + dt * z2) / denom_3d
        divp = p1 - p1[:, il, :] + p2 - p2[iux, :, :]

    u = x - divp / _lambda
    return u

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`

In [ ]:
def forward_operator(x, Phi):
    """
    Forward model of snapshot compressive imaging (SCI).
    Multiple encoded frames are collapsed into a single measurement.
    Phi: Sensing matrix (Mask)
    """
    return np.sum(x * Phi, axis=2)

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `run_inversion`

In [ ]:
def run_inversion(meas, Phi, truth, _lambda=1, iter_max=20, tv_weight=6, tv_iter_max=5, step=1):
    """
    Generalized Alternating Projection (GAP) for SCI reconstruction.
    Using TV as the denoiser.
    """
    # Initialization
    x = backward_operator(meas, Phi)  # A^T(y)
    y1 = np.zeros_like(meas)

    Phi_sum = np.sum(Phi, 2)
    Phi_sum[Phi_sum == 0] = 1

    psnr_all = []

    print(f"Starting GAP-TV Reconstruction for {iter_max} iterations...")

    for k in range(iter_max):
        # 1. Data Projection (GAP)
        yb = forward_operator(x, Phi)
        y1 = y1 + (meas - yb)
        x = x + _lambda * (backward_operator((y1 - yb) / Phi_sum, Phi))

        # 2. Denoising (Prior)
        # Shift back to image domain for denoising
        x_img = shift_back(x, step=step)

        # TV Denoising
        x_img = TV_denoiser(x_img, tv_weight, n_iter_max=tv_iter_max)

        # Shift forward to measurement domain
        x = shift(x_img, step=step)

        # Metrics
        if truth is not None:
            current_psnr = psnr(truth, x_img)
            psnr_all.append(current_psnr)
            if (k + 1) % 1 == 0:
                print(f"  Iteration {k+1}/{iter_max}, PSNR: {current_psnr:.2f} dB")

    return x_img, psnr_all

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `psnr`, `evaluate_results`

In [ ]:
def psnr(ref, img):
    """
    Peak signal-to-noise ratio (PSNR).
    """
    mse = np.mean((ref - img) ** 2)
    if mse == 0:
        return 100
    PIXEL_MAX = 1.0
    return 20 * math.log10(PIXEL_MAX / math.sqrt(mse))

def evaluate_results(recon_img, truth, psnrs, output_dir='.'):
    """
    Evaluate and save reconstruction results.
    """
    # Final PSNR
    final_psnr = psnr(truth, recon_img)
    print(f"Final PSNR: {final_psnr:.2f} dB")

    # Save Reconstruction as .mat
    sio.savemat(os.path.join(output_dir, 'recon_result.mat'), {'img': recon_img})

    # Save spectral channels as image grid
    nC = recon_img.shape[2]
    fig = plt.figure(figsize=(10, 10))
    for i in range(min(9, nC)):
        plt.subplot(3, 3, i + 1)
        plt.imshow(recon_img[:, :, i], cmap='gray', vmin=0, vmax=1)
        plt.axis('off')
        plt.title(f'Band {i+1}')
    plt.savefig(os.path.join(output_dir, 'recon_channels.png'))
    plt.close()

    # Save PSNR plot
    plt.figure()
    plt.plot(psnrs)
    plt.xlabel('Iteration')
    plt.ylabel('PSNR (dB)')
    plt.title('Reconstruction Convergence')
    plt.savefig(os.path.join(output_dir, 'psnr_curve.png'))
    plt.close()

    return final_psnr

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Configuration
datname = 'kaist_crop256_01'
matfile = f'Dataset/{datname}.mat'
maskfile = 'mask/mask256.mat'
step = 1
n_iter = 5

if not os.path.exists(matfile) or not os.path.exists(maskfile):
    print(f"Error: Data files not found. Please ensure {matfile} and {maskfile} exist.")
    exit(1)

### 1. Load and preprocess data

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 1. Load and preprocess data
print("Loading data...")
truth, mask_3d, meas = load_and_preprocess_data(matfile, maskfile, step=step)

# Save Measurement
plt.imsave('measurement.png', meas, cmap='gray')

### 2. Run inversion (GAP-TV reconstruction)

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 2. Run inversion (GAP-TV reconstruction)
print(f"Running reconstruction (GAP-TV) with {n_iter} iterations...")
recon_img, psnrs = run_inversion(
    meas,
    mask_3d,
    truth,
    _lambda=1,
    iter_max=n_iter,
    tv_weight=6,
    tv_iter_max=5,
    step=step
)

### 3. Evaluate results

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Evaluate results
print("Saving results...")
final_psnr = evaluate_results(recon_img, truth, psnrs, output_dir='.')

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pnp_cassi**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=21.76 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Deep plug-and-play priors for spectral snapshot compressive imaging
- Repository: https://github.com/zsm1211/PnP-CASSI
